# Results viewer — read whatever is already on disk

### VERSION 2 — rare cell-type metrics + significance test included

If the version line printed by the config cell below does not say **v2**, you are running a stale copy: close the notebook, let Drive sync, and reopen. v1 had status + Table-1 metrics only; v2 adds the dependency-free rare-cell metrics and the paired Wilcoxon test at the end.

**Read-only.** Nothing is trained, no model is loaded, no file is written. It scans the model directory on Drive and prints every metric that has been saved so far.

**No installs needed.** Only `pandas`, `numpy` and the standard library — all preinstalled on Colab. Do *not* run the pip cell from the other notebooks here. Set the runtime to **CPU** (Runtime → Change runtime type) so this doesn't compete for the GPU with a training session.

Safe to run in a second Colab tab while another notebook is still training: it only reads.

Start with the **status** section — it shows how far a run has got (which stages exist on disk for each dataset), which is the fastest way to see what is finished and what is still in flight.

In [24]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
import os, json, glob
import numpy as np
import pandas as pd

VIEWER_VERSION = 'v2 — status + Table-1 metrics + rare-cell metrics + Wilcoxon significance'
print('=' * 78)
print('RESULTS VIEWER', VIEWER_VERSION)
print('=' * 78)

MODEL_DIR = '/content/drive/MyDrive/models'
DATASETS_TO_SHOW = ['pancreas', 'lung', 'pbmc-immune']
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 60)

print(MODEL_DIR, '->', 'OK' if os.path.isdir(MODEL_DIR) else 'NOT FOUND')
for ds in DATASETS_TO_SHOW:
    d = os.path.join(MODEL_DIR, ds)
    print(f"  {ds:<14} {'OK' if os.path.isdir(d) else 'missing'}")

RESULTS VIEWER v2 — status + Table-1 metrics + rare-cell metrics + Wilcoxon significance
/content/drive/MyDrive/models -> OK
  pancreas       OK
  lung           OK
  pbmc-immune    OK


## Status — what has finished, per dataset

In [26]:
def stage_status(ds):
    # Which artefacts of the scVI Stage-2 experiment exist on disk for this dataset.
    base = os.path.join(MODEL_DIR, ds)
    rows = []

    scvi_dirs = glob.glob(os.path.join(base, 'scvi_stage1', '*', 'model.pt'))
    rows.append(('Stage 1: scVI weights', len(scvi_dirs) > 0,
                 os.path.dirname(scvi_dirs[0]) if scvi_dirs else ''))

    pre = glob.glob(os.path.join(base, 'pretrain_scvi', '*', 'pretrain_checkpoint.pth'))
    rows.append(('Stage 1: model checkpoint', len(pre) > 0,
                 os.path.dirname(pre[0]) if pre else ''))

    sea = os.path.join(base, 'seacell_X_scvidef', 'metrics.json')
    rows.append(('arm A: SEACells (scVI)', os.path.exists(sea), sea if os.path.exists(sea) else ''))

    lei = glob.glob(os.path.join(base, 'leiden_X_scvidef_K*', 'metrics.json'))
    rows.append(('arm A: Leiden (scVI)', len(lei) > 0, lei[0] if lei else ''))

    s2 = glob.glob(os.path.join(base, 'scviproto_*'))
    ckpt = [d for d in s2 if os.path.exists(os.path.join(d, 'umap_checkpoint.pth'))]
    mets = [d for d in s2 if os.path.exists(os.path.join(d, 'metrics.json'))]
    cells_csv = [d for d in s2 if os.path.exists(os.path.join(d, 'umap_cells.csv'))]
    rows.append(('arm B: Stage-2 checkpoint', len(ckpt) > 0, ckpt[0] if ckpt else ''))
    rows.append(('arm B: metrics.json (eval done)', len(mets) > 0, mets[0] if mets else ''))
    rows.append(('arm B: umap_cells.csv (rare table ready)', len(cells_csv) > 0, cells_csv[0] if cells_csv else ''))

    df = pd.DataFrame(rows, columns=['stage', 'done', 'path'])
    df['done'] = df['done'].map({True: 'YES', False: '--'})
    return df

for ds in DATASETS_TO_SHOW:
    print(f"\n=== {ds} ===")
    display(stage_status(ds))


=== pancreas ===


,stage,done,path
0,Stage 1: scVI weights,YES,/content/drive/MyDrive/models/pancreas/scvi_stage1/d10_z...
1,Stage 1: model checkpoint,YES,/content/drive/MyDrive/models/pancreas/pretrain_scvi/d10...
2,arm A: SEACells (scVI),YES,/content/drive/MyDrive/models/pancreas/seacell_X_scvidef...
3,arm A: Leiden (scVI),YES,/content/drive/MyDrive/models/pancreas/leiden_X_scvidef_...
4,arm B: Stage-2 checkpoint,YES,/content/drive/MyDrive/models/pancreas/scviproto_ds-panc...
5,arm B: metrics.json (eval done),YES,/content/drive/MyDrive/models/pancreas/scviproto_ds-panc...
6,arm B: umap_cells.csv (rare table ready),YES,/content/drive/MyDrive/models/pancreas/scviproto_ds-panc...



=== lung ===


,stage,done,path
0,Stage 1: scVI weights,YES,/content/drive/MyDrive/models/lung/scvi_stage1/d10_zinb_e50
1,Stage 1: model checkpoint,YES,/content/drive/MyDrive/models/lung/pretrain_scvi/d10_zin...
2,arm A: SEACells (scVI),YES,/content/drive/MyDrive/models/lung/seacell_X_scvidef/met...
3,arm A: Leiden (scVI),YES,/content/drive/MyDrive/models/lung/leiden_X_scvidef_K300...
4,arm B: Stage-2 checkpoint,YES,/content/drive/MyDrive/models/lung/scviproto_ds-lung_LD1...
5,arm B: metrics.json (eval done),YES,/content/drive/MyDrive/models/lung/scviproto_ds-lung_LD1...
6,arm B: umap_cells.csv (rare table ready),YES,/content/drive/MyDrive/models/lung/scviproto_ds-lung_LD1...



=== pbmc-immune ===


,stage,done,path
0,Stage 1: scVI weights,YES,/content/drive/MyDrive/models/pbmc-immune/scvi_stage1/d1...
1,Stage 1: model checkpoint,YES,/content/drive/MyDrive/models/pbmc-immune/pretrain_scvi/...
2,arm A: SEACells (scVI),--,
3,arm A: Leiden (scVI),--,
4,arm B: Stage-2 checkpoint,--,
5,arm B: metrics.json (eval done),--,
6,arm B: umap_cells.csv (rare table ready),--,


## Every saved metric, per dataset

One row per run directory that has a `metrics.json`. This includes runs from all the other experiments in the model dir, not just this one — filter below.

In [27]:
KEY_METRICS = [
    'mean_cell_type_purity', 'mean_batch_entropy', 'coverage',
    'modularity', 'mean_modularity_batch', 'std_modularity_batch',
    'n_unused_protos', 'K_target', 'latent_dim', 'stage1',
    'scgraph_corr_avg', 'dge_rbo_avg',
]

def load_all_metrics(ds):
    base = os.path.join(MODEL_DIR, ds)
    if not os.path.isdir(base):
        return pd.DataFrame()
    rows = []
    for entry in sorted(os.listdir(base)):
        path = os.path.join(base, entry, 'metrics.json')
        if not os.path.exists(path):
            continue
        try:
            with open(path) as f:
                m = json.load(f)
        except Exception as e:
            print(f"  could not read {path}: {e}")
            continue
        row = {'dataset': ds, 'run': entry,
               'modified': pd.to_datetime(os.path.getmtime(path), unit='s')}
        row.update({k: m.get(k) for k in KEY_METRICS})
        rows.append(row)
    return pd.DataFrame(rows)

df_all = pd.concat([load_all_metrics(ds) for ds in DATASETS_TO_SHOW], ignore_index=True)
print(f"{len(df_all)} runs with metrics.json")
df_all.sort_values(['dataset', 'modified'], ascending=[True, False]).head(60)

231 runs with metrics.json


/tmp/ipykernel_1523/2755541311.py:29: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all = pd.concat([load_all_metrics(ds) for ds in DATASETS_TO_SHOW], ignore_index=True)


,dataset,run,modified,mean_cell_type_purity,mean_batch_entropy,coverage,modularity,mean_modularity_batch,std_modularity_batch,n_unused_protos,K_target,latent_dim,stage1,scgraph_corr_avg,dge_rbo_avg
117,lung,leiden_X_harmony_d20_K300,2026-08-04 10:12:12,0.846107,1.091180,0.882353,0.562943,0.513888,0.111389,0.0,300.0,NaN,None,NaN,NaN
159,lung,seacell_X_harmony_d20,2026-08-04 09:59:28,0.794837,1.328981,1.000000,0.328144,0.300543,0.072031,0.0,300.0,NaN,None,0.865363,NaN
152,lung,scviproto_ds-lung_LD10_prtInit-wayp_aff-arbf_lprec0.01_u...,2026-08-04 09:25:16,0.872173,0.558935,1.000000,0.727400,0.650069,0.039418,3.0,300.0,10.0,scvi,0.867028,0.068047
122,lung,leiden_X_scvidef_K300,2026-08-04 09:15:23,0.850151,1.185238,1.000000,0.575166,0.522427,0.077166,0.0,300.0,NaN,None,NaN,NaN
164,lung,seacell_X_scvidef,2026-08-04 09:08:40,0.860817,1.110295,1.000000,0.343097,0.326896,0.052737,0.0,300.0,NaN,None,0.829660,NaN
145,lung,proto_umap_ds-lung_LD50_prtInit-wayp_aff-arbf_lprec0.01_...,2026-08-04 08:12:13,0.867851,0.526079,1.000000,0.716116,0.653985,0.027298,2.0,NaN,NaN,None,0.932549,0.059203
118,lung,leiden_X_harmony_d50_K300,2026-08-04 08:05:10,0.863552,0.979055,1.000000,0.672415,0.569906,0.070164,0.0,300.0,NaN,None,NaN,NaN
160,lung,seacell_X_harmony_d50,2026-08-04 08:00:06,0.823471,1.386134,0.941176,0.357445,0.323474,0.060276,0.0,300.0,NaN,None,0.865218,NaN
149,lung,proto_umap_ds-lung_prtInit-wayp_aff-arbf_lprec0.01_usim-...,2026-08-02 20:00:19,0.858799,0.537913,0.882353,0.728414,0.664444,0.023622,2.0,NaN,NaN,None,0.868208,0.067389
141,lung,metacell2,2026-07-31 13:00:02,0.900350,0.709903,1.000000,0.246565,0.227241,0.059200,0.0,300.0,NaN,None,0.660855,NaN


## The scVI Stage-2 experiment — the three arms

`scviproto_*` = scProto Stage 2 on scVI, `seacell_X_scvidef` / `leiden_X_scvidef_K*` = the two-step baselines on the same scVI latent. The canonical `proto_umap_*` run (scPoli Stage 1) is included as context.

In [28]:
def label_run(run):
    if run.startswith('scviproto'):            return 'scProto Stage 2 (scVI)'
    if run == 'seacell_X_scvidef':             return 'SEACells (scVI)'
    if run.startswith('leiden_X_scvidef_K'):   return 'Leiden (scVI)'
    if run.startswith('leiden_X_scvil2_K'):    return 'Leiden (scVI, L2-norm control)'
    if run.startswith('proto_umap'):           return 'scProto (scPoli Stage 1)'
    if run == 'seacell':                       return 'SEACells (PCA)'
    return None

arms = df_all.copy()
arms['method'] = arms['run'].map(label_run)
arms = arms[arms['method'].notna()]

show = arms[['dataset', 'method', 'mean_modularity_batch', 'std_modularity_batch',
             'mean_cell_type_purity', 'mean_batch_entropy', 'coverage', 'K_target', 'latent_dim']]
show = show.rename(columns={'mean_modularity_batch': 'modularity',
                            'std_modularity_batch': 'modularity_std',
                            'mean_cell_type_purity': 'purity',
                            'mean_batch_entropy': 'batch_entropy'})
display(show.sort_values(['dataset', 'method']).set_index(['dataset', 'method']).round(3))

modularity  modularity_std  purity  batch_entropy  coverage  K_target  latent_dim
dataset     method                                                                                                     
lung        Leiden (scVI)                  0.522           0.077   0.850          1.185     1.000     300.0         NaN
            SEACells (PCA)                 0.674           0.039   0.905          0.443     1.000       NaN         NaN
            SEACells (scVI)                0.327           0.053   0.861          1.110     1.000     300.0         NaN
            scProto (scPoli Stage 1)       0.654           0.027   0.868          0.526     1.000       NaN         NaN
            scProto (scPoli Stage 1)       0.654           0.052   0.929          0.613     0.941       NaN         NaN
            scProto (scPoli Stage 1)       0.670           0.031   0.880          0.503     0.941       NaN         NaN
            scProto (scPoli Stage 1)       0.644           0.036   0.892          0.544     0.941       NaN         NaN
            scProto (scPoli Stage 1)       0.664           0.024   0.859          0.538     0.882       NaN         NaN
            scProto (scPoli Stage 1)       0.692           0.032   0.873          0.459     0.941       NaN         NaN
            scProto Stage 2 (scVI)         0.650           0.039   0.872          0.559     1.000     300.0        10.0
pancreas    Leiden (scVI)                  0.379           0.122   0.965          0.770     0.786     220.0         NaN
            SEACells (PCA)                 0.674           0.054   0.956          0.179     0.857       NaN         NaN
            SEACells (scVI)                0.296           0.078   0.955          0.810     1.000     220.0         NaN
            scProto (scPoli Stage 1)       0.615           0.069   0.916          0.389     0.929       NaN         NaN
            scProto (scPoli Stage 1)       0.611           0.082   0.916          0.365     0.929       NaN         NaN
            scProto (scPoli Stage 1)       0.477           0.040   0.962          0.352     0.857       NaN         NaN
            scProto (scPoli Stage 1)       0.592           0.081   0.915          0.400     1.000       NaN         NaN
            scProto (scPoli Stage 1)       0.482           0.045   0.961          0.349     0.786       NaN         NaN
            scProto (scPoli Stage 1)       0.473           0.042   0.956          0.384     0.857       NaN         NaN
            scProto (scPoli Stage 1)       0.601           0.088   0.913          0.399     0.929       NaN         NaN
            scProto (scPoli Stage 1)       0.637           0.089   0.942          0.326     0.929       NaN         NaN
            scProto (scPoli Stage 1)         NaN             NaN   0.910          0.440     1.000       NaN         NaN
            scProto (scPoli Stage 1)         NaN             NaN   0.908          0.860     0.929       NaN         NaN
            scProto (scPoli Stage 1)         NaN             NaN   0.910          0.844     0.929       NaN         NaN
            scProto (scPoli Stage 1)         NaN             NaN   0.944          0.321     0.929       NaN         NaN
            scProto (scPoli Stage 1)         NaN             NaN   0.906          0.850     0.929       NaN         NaN
            scProto (scPoli Stage 1)         NaN             NaN     NaN            NaN       NaN       NaN         NaN
            scProto Stage 2 (scVI)         0.616           0.082   0.914          0.301     1.000     220.0        10.0
pbmc-immune SEACells (PCA)                 0.554           0.035   0.894          0.135     1.000       NaN         NaN
            scProto (scPoli Stage 1)       0.631           0.054   0.878          0.222     1.000       NaN         NaN
            scProto (scPoli Stage 1)         NaN             NaN     NaN            NaN       NaN       NaN         NaN
            scProto (scPoli Stage 1)         NaN             NaN   0.871      

In [29]:
# Modularity side by side -- the headline comparison.
pivot = show.pivot_table(index='method', columns='dataset', values='modularity', aggfunc='first')
display(pivot.round(3))

dataset,lung,pancreas,pbmc-immune
method,,,
Leiden (scVI),0.522,0.379,NaN
SEACells (PCA),0.674,0.674,0.554
SEACells (scVI),0.327,0.296,NaN
scProto (scPoli Stage 1),0.654,0.615,0.631
scProto Stage 2 (scVI),0.650,0.616,NaN


## Stage-2 training state

`stage2_state.json` records how many epochs a Stage-2 run trained and whether it stopped early (converged) or hit its epoch cap.

In [30]:
rows = []
for ds in DATASETS_TO_SHOW:
    for d in sorted(glob.glob(os.path.join(MODEL_DIR, ds, 'scviproto_*'))):
        p = os.path.join(d, 'stage2_state.json')
        r = {'dataset': ds, 'run': os.path.basename(d)}
        if os.path.exists(p):
            r.update(json.load(open(p)))
        else:
            r['note'] = 'no state file (run predates this bookkeeping)'
        rows.append(r)
pd.DataFrame(rows) if rows else print("no scviproto_* runs found yet")

,dataset,run,note,epochs_trained,max_epochs_requested,hit_epoch_cap
0,pancreas,scviproto_ds-panc_NP220_LD10_prtInit-wayp_aff-arbf_lprec...,no state file (run predates this bookkeeping),NaN,NaN,NaN
1,lung,scviproto_ds-lung_LD10_prtInit-wayp_aff-arbf_lprec0.01_u...,NaN,9.0,20.0,False
2,pbmc-immune,scviproto_LD10_prtInit-wayp_aff-arbf_lprec0.01_usim-prot...,no state file (run predates this bookkeeping),NaN,NaN,NaN


## Rare-cell metrics (optional)

Needs the full environment (`scanpy`/`scarches`/`nb_setup`), so it is skipped automatically on a plain CPU runtime. Run it in a session where the other notebook's install cell has already run.

In [31]:
try:
    import sys
    PROJECT_ROOT = '/content/drive/MyDrive/codes/interpretable-prototype'
    if PROJECT_ROOT not in sys.path:
        sys.path.insert(0, PROJECT_ROOT)
    os.environ.setdefault('MODEL_DIR', MODEL_DIR + '/')
    os.environ.setdefault('DATA_DIR', '/content/drive/MyDrive/data/')
    os.environ.setdefault('CODE_DIR', PROJECT_ROOT + '/')
    os.environ.setdefault('HOME_DIR', '/content/drive/MyDrive/')

    from interpretable_ssl.evaluation.paper_figures import (
        rare_celltype_purity_table, rare_metric_significance_paired,
    )
    from interpretable_ssl.experiments.scvi_stage2 import resolve_run_names
    HAVE_ENV = True
except Exception as e:
    HAVE_ENV = False
    print(f"full environment not available here ({type(e).__name__}: {e})\n"
          f"-> skip this section, or run it in a runtime where the install cell has run.")

full environment not available here (ModuleNotFoundError: No module named 'anndata')
-> skip this section, or run it in a runtime where the install cell has run.


In [32]:
if HAVE_ENV:
    stage2_dirs = {ds: sorted(glob.glob(os.path.join(MODEL_DIR, ds, 'scviproto_*')))
                   for ds in DATASETS_TO_SHOW}
    first = next((v[0] for v in stage2_dirs.values() if v), None)
    if first is None:
        print("no scviproto_* run on disk yet -- nothing to build the keyword set from")
    else:
        MODEL_KEYWORDS = resolve_run_names(first, include_canonical=True)
        ready = [ds for ds in DATASETS_TO_SHOW
                 if glob.glob(os.path.join(MODEL_DIR, ds, 'scviproto_*', 'umap_cells.csv'))]
        print("datasets with a finished Stage-2 eval:", ready)
        if ready:
            df_rare = rare_celltype_purity_table(ready, model_keywords=MODEL_KEYWORDS, verbose=False)
            display(df_rare[[c for c in df_rare.columns if not c.startswith('_')]].round(3))

In [33]:
if HAVE_ENV and 'df_rare' in dir():
    for metric in ['batch_rare_f1_macro', 'batch_rare_homogeneity']:
        rows = []
        for (ds, run) in df_rare.index:
            vals = np.asarray(df_rare.loc[(ds, run), f'_{metric}_per_batch'], dtype=float)
            rows.append({'dataset': ds, 'method': run, 'n_batches': len(vals),
                         'mean': round(float(vals.mean()), 3),
                         'std': round(float(vals.std()), 3)})
        print(f"=== {metric} (mean ± std across batches) ===")
        display(pd.DataFrame(rows).pivot(index='method', columns='dataset', values=['mean', 'std']))

## Rare cell-type metrics + significance test — no heavy dependencies

The section above needs `scanpy`/`scarches`. This one does not: it reads each run's `umap_cells.csv` (cell_id, metacell_id, label, batch) and recomputes the paper's own rare-cell metrics with pandas/numpy/scipy only, so it works on a plain CPU runtime while training is still going.

Definitions are ported verbatim from `paper_figures._rare_table_one`, so these numbers are the same quantity as every other rare-cell table in the rebuttal:

- **locally-rare types**, per batch: within-batch frequency below `mean - std`, falling back to the 25th percentile
- **recall** = fraction of a type's cells in that batch whose metacell has that type as its majority label
- **precision** = mean purity of the metacells dedicated to that type (global)
- **F1** = harmonic mean of the two, macro-averaged over the batch's rare types
- **homogeneity** = mean fraction of a rare cell's metacell that shares its label
- **cross-batch homogeneity** = same, counting only same-label mates from a *different* batch

In [34]:
# label/batch column per dataset (hardcoded so this notebook needs no project imports)
LABEL_BATCH = {
    'pancreas':    ('celltype', 'tech'),
    'lung':        ('cell_type', 'batch'),
    'pbmc-immune': ('final_annotation', 'study'),
}

# The paper's canonical scProto runs (scPoli Stage 1), by exact folder name -- a
# 'proto_umap' prefix also matches lambda-sweep and d=50 variants sitting in the same
# directory, which would silently become extra 'scProto' rows.
CANONICAL_SCPROTO = {
    'pancreas': 'proto_umap_ds-panc_NP220_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31',
    'lung':     'proto_umap_ds-lung_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31',
    'pbmc-immune': 'proto_umap_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31',
}

def arm_dirs(ds):
    # {display name: run dir} for the arms of this experiment, only those with umap_cells.csv
    base = os.path.join(MODEL_DIR, ds)
    out = {}
    for d in sorted(glob.glob(os.path.join(base, 'scviproto_*'))):
        out['scProto Stage 2 (scVI)'] = d
    p = os.path.join(base, 'seacell_X_scvidef')
    if os.path.isdir(p): out['SEACells (scVI)'] = p
    for d in sorted(glob.glob(os.path.join(base, 'leiden_X_scvidef_K*'))):
        out['Leiden (scVI)'] = d
    canon = os.path.join(base, CANONICAL_SCPROTO.get(ds, '_none_'))
    if os.path.isdir(canon): out['scProto (scPoli Stage 1)'] = canon
    return {k: v for k, v in out.items() if os.path.exists(os.path.join(v, 'umap_cells.csv'))}

for ds in DATASETS_TO_SHOW:
    print(ds, '->', list(arm_dirs(ds)))

pancreas -> ['scProto Stage 2 (scVI)', 'SEACells (scVI)', 'Leiden (scVI)', 'scProto (scPoli Stage 1)']
lung -> ['scProto Stage 2 (scVI)', 'SEACells (scVI)', 'Leiden (scVI)', 'scProto (scPoli Stage 1)']
pbmc-immune -> ['scProto (scPoli Stage 1)']


In [35]:
def rare_metrics_per_batch(cells_df, label_key, batch_key, rare_quantile=0.25):
    # Verbatim port of paper_figures._rare_table_one's batch-local rare metrics.
    mc_sizes        = cells_df.groupby('metacell_id').size()
    mc_label_counts = cells_df.groupby(['metacell_id', label_key]).size()
    mc_label_frac   = mc_label_counts / mc_sizes
    mc_majority     = mc_label_counts.groupby(level='metacell_id').idxmax().map(lambda x: x[1])

    mc_purity_series = mc_label_frac.reindex(
        pd.MultiIndex.from_arrays([mc_majority.index, mc_majority.values])).fillna(0.0)
    mc_purity_series.index = mc_majority.index
    mc_purity_by_ct = mc_purity_series.groupby(mc_majority).mean().to_dict()
    mc_label_batch_counts = cells_df.groupby(['metacell_id', label_key, batch_key]).size()

    out = {'f1': [], 'homogeneity': [], 'cross_batch_homog': [], 'coverage': [],
           'recall': [], 'precision': []}

    for batch_val, batch_df in cells_df.groupby(batch_key):
        ct_freq = batch_df[label_key].value_counts(normalize=True)
        thr = float(ct_freq.mean() - ct_freq.std())
        bl_rare = set(ct_freq[ct_freq < thr].index)
        if not bl_rare:
            bl_rare = set(ct_freq[ct_freq < ct_freq.quantile(rare_quantile)].index)
        if not bl_rare:
            continue
        bl_cells = batch_df[batch_df[label_key].isin(bl_rare)]
        if bl_cells.empty:
            continue

        n_cov = sum((batch_df.loc[batch_df[label_key] == ct, 'metacell_id'].map(mc_majority) == ct).any()
                    for ct in bl_rare)
        out['coverage'].append(n_cov / len(bl_rare))

        lookup = pd.MultiIndex.from_arrays([bl_cells['metacell_id'], bl_cells[label_key]])
        scores = mc_label_frac.reindex(lookup).fillna(0.0)
        out['homogeneity'].append(float(scores.mean()))

        sb = pd.MultiIndex.from_arrays([bl_cells['metacell_id'], bl_cells[label_key],
                                        pd.Index([batch_val] * len(bl_cells))])
        same_batch = mc_label_batch_counts.reindex(sb).fillna(0.0).values
        total_same = mc_label_counts.reindex(lookup).fillna(0.0).values
        cross = (total_same - same_batch) / mc_sizes.reindex(bl_cells['metacell_id']).values
        out['cross_batch_homog'].append(float(np.mean(cross)))

        f1s, recs, precs = [], [], []
        for ct in bl_rare:
            ct_cells = batch_df[batch_df[label_key] == ct]
            if ct_cells.empty:
                continue
            r = float((ct_cells['metacell_id'].map(mc_majority) == ct).mean())
            p = mc_purity_by_ct.get(ct, 0.0)
            f1s.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
            recs.append(r); precs.append(p)
        if f1s:
            out['f1'].append(float(np.mean(f1s)))
            out['recall'].append(float(np.mean(recs)))
            out['precision'].append(float(np.mean(precs)))
    return out


def load_cells(run_dir, label_key, batch_key):
    df = pd.read_csv(os.path.join(run_dir, 'umap_cells.csv'),
                     usecols=lambda c: c not in ('umap_1', 'umap_2'))
    missing = [c for c in (label_key, batch_key, 'metacell_id') if c not in df.columns]
    if missing:
        raise KeyError(f"{run_dir}: umap_cells.csv missing {missing}; has {list(df.columns)}")
    return df

RARE = {}   # (dataset, method) -> per-batch dict
for ds in DATASETS_TO_SHOW:
    lk, bk = LABEL_BATCH[ds]
    for name, d in arm_dirs(ds).items():
        try:
            RARE[(ds, name)] = rare_metrics_per_batch(load_cells(d, lk, bk), lk, bk)
        except Exception as e:
            print(f"  [{ds}] {name}: {type(e).__name__}: {e}")
print(f"computed rare metrics for {len(RARE)} (dataset, method) pairs")

computed rare metrics for 9 (dataset, method) pairs


In [36]:
rows = []
for (ds, name), m in RARE.items():
    row = {'dataset': ds, 'method': name}
    for k, v in m.items():
        row[k] = f"{np.mean(v):.3f} ± {np.std(v):.3f}" if v else None
    row['n_batches'] = len(m['f1'])
    rows.append(row)
df_rare_view = pd.DataFrame(rows).set_index(['dataset', 'method']).sort_index()
display(df_rare_view[['f1', 'homogeneity', 'cross_batch_homog', 'coverage',
                      'recall', 'precision', 'n_batches']])

f1    homogeneity cross_batch_homog       coverage         recall      precision  n_batches
dataset     method                                                                                                                          
lung        Leiden (scVI)             0.689 ± 0.167  0.635 ± 0.182     0.597 ± 0.159  0.856 ± 0.181  0.718 ± 0.179  0.812 ± 0.067         15
            SEACells (scVI)           0.682 ± 0.162  0.658 ± 0.180     0.598 ± 0.154  0.856 ± 0.181  0.712 ± 0.172  0.810 ± 0.095         15
            scProto (scPoli Stage 1)  0.599 ± 0.176  0.578 ± 0.221     0.508 ± 0.197  0.789 ± 0.177  0.604 ± 0.228  0.702 ± 0.137         15
            scProto Stage 2 (scVI)    0.557 ± 0.255  0.560 ± 0.257     0.466 ± 0.242  0.706 ± 0.303  0.555 ± 0.272  0.853 ± 0.026         15
pancreas    Leiden (scVI)             0.258 ± 0.276  0.317 ± 0.242     0.237 ± 0.146  0.302 ± 0.291  0.294 ± 0.273  0.332 ± 0.257          8
            SEACells (scVI)           0.639 ± 0.196  0.588 ± 0.180     0.411 ± 0.123  0.885 ± 0.224  0.720 ± 0.258  0.722 ± 0.112          8
            scProto (scPoli Stage 1)  0.535 ± 0.208  0.565 ± 0.155     0.323 ± 0.179  0.635 ± 0.235  0.537 ± 0.204  0.768 ± 0.133          8
            scProto Stage 2 (scVI)    0.614 ± 0.218  0.601 ± 0.173     0.205 ± 0.133  0.729 ± 0.246  0.592 ± 0.208  0.919 ± 0.027          8
pbmc-immune scProto (scPoli Stage 1)  0.854 ± 0.138  0.827 ± 0.102     0.349 ± 0.198  0.933 ± 0.133  0.880 ± 0.118  0.903 ± 0.034          5

In [37]:
# Paired one-sided Wilcoxon (scProto Stage 2 > baseline) across batches,
# Bonferroni-corrected per dataset and metric -- same convention as the paper's tables.
from scipy.stats import wilcoxon

REF = 'scProto Stage 2 (scVI)'

def stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

sig_rows = []
for ds in DATASETS_TO_SHOW:
    methods = [m for (d, m) in RARE if d == ds]
    if REF not in methods:
        continue
    others = [m for m in methods if m != REF]
    for metric in ['f1', 'homogeneity', 'cross_batch_homog']:
        ref = np.asarray(RARE[(ds, REF)][metric], dtype=float)
        for m in methods:
            arr = np.asarray(RARE[(ds, m)][metric], dtype=float)
            r = {'dataset': ds, 'metric': metric, 'method': m, 'n': len(arr),
                 'mean': round(float(arr.mean()), 3) if len(arr) else np.nan,
                 'std': round(float(arr.std()), 3) if len(arr) else np.nan}
            if m != REF and len(arr) == len(ref) and len(arr):
                r['n_wins'] = int((ref > arr).sum())
                try:
                    _, p = wilcoxon(ref, arr, alternative='greater')
                    r['p_adj'] = round(min(float(p) * max(len(others), 1), 1.0), 4)
                    r['sig'] = stars(r['p_adj'])
                except ValueError:
                    pass
            sig_rows.append(r)

df_sig_view = pd.DataFrame(sig_rows)
for metric in ['f1', 'homogeneity', 'cross_batch_homog']:
    sub = df_sig_view[df_sig_view['metric'] == metric].copy()
    if sub.empty:
        continue
    sub['cell'] = sub.apply(
        lambda r: f"{r['mean']:.3f}±{r['std']:.3f} (n={r['n']}) [ref]" if r['method'] == REF
        else f"{r['mean']:.3f}±{r['std']:.3f} (wins={r.get('n_wins','?')}/{r['n']})  "
             f"{r.get('sig','?')}  p_adj={r.get('p_adj', float('nan')):.3g}", axis=1)
    print(f"=== rare {metric}: {REF} vs each arm, paired one-sided Wilcoxon ===")
    display(sub.pivot(index='method', columns='dataset', values='cell'))

df_sig_view

=== rare f1: scProto Stage 2 (scVI) vs each arm, paired one-sided Wilcoxon ===


dataset,lung,pancreas
method,,
Leiden (scVI),0.689±0.167 (wins=4.0/15) ns p_adj=1,0.258±0.276 (wins=8.0/8) * p_adj=0.0117
SEACells (scVI),0.682±0.162 (wins=1.0/15) ns p_adj=1,0.639±0.196 (wins=3.0/8) ns p_adj=1
scProto (scPoli Stage 1),0.599±0.176 (wins=8.0/15) ns p_adj=1,0.535±0.208 (wins=6.0/8) ns p_adj=0.469
scProto Stage 2 (scVI),0.557±0.255 (n=15) [ref],0.614±0.218 (n=8) [ref]


=== rare homogeneity: scProto Stage 2 (scVI) vs each arm, paired one-sided Wilcoxon ===


dataset,lung,pancreas
method,,
Leiden (scVI),0.635±0.182 (wins=4.0/15) ns p_adj=1,0.317±0.242 (wins=8.0/8) * p_adj=0.0117
SEACells (scVI),0.658±0.180 (wins=2.0/15) ns p_adj=1,0.588±0.180 (wins=3.0/8) ns p_adj=1
scProto (scPoli Stage 1),0.578±0.221 (wins=7.0/15) ns p_adj=1,0.565±0.155 (wins=5.0/8) ns p_adj=0.691
scProto Stage 2 (scVI),0.560±0.257 (n=15) [ref],0.601±0.173 (n=8) [ref]


=== rare cross_batch_homog: scProto Stage 2 (scVI) vs each arm, paired one-sided Wilcoxon ===


dataset,lung,pancreas
method,,
Leiden (scVI),0.597±0.159 (wins=3.0/15) ns p_adj=1,0.237±0.146 (wins=4.0/8) ns p_adj=1
SEACells (scVI),0.598±0.154 (wins=3.0/15) ns p_adj=1,0.411±0.123 (wins=0.0/8) ns p_adj=1
scProto (scPoli Stage 1),0.508±0.197 (wins=6.0/15) ns p_adj=1,0.323±0.179 (wins=2.0/8) ns p_adj=1
scProto Stage 2 (scVI),0.466±0.242 (n=15) [ref],0.205±0.133 (n=8) [ref]


,dataset,metric,method,n,mean,std,n_wins,p_adj,sig
0,pancreas,f1,scProto Stage 2 (scVI),8,0.614,0.218,NaN,NaN,NaN
1,pancreas,f1,SEACells (scVI),8,0.639,0.196,3.0,1.0000,ns
2,pancreas,f1,Leiden (scVI),8,0.258,0.276,8.0,0.0117,*
3,pancreas,f1,scProto (scPoli Stage 1),8,0.535,0.208,6.0,0.4688,ns
4,pancreas,homogeneity,scProto Stage 2 (scVI),8,0.601,0.173,NaN,NaN,NaN
5,pancreas,homogeneity,SEACells (scVI),8,0.588,0.180,3.0,1.0000,ns
6,pancreas,homogeneity,Leiden (scVI),8,0.317,0.242,8.0,0.0117,*
7,pancreas,homogeneity,scProto (scPoli Stage 1),8,0.565,0.155,5.0,0.6914,ns
8,pancreas,cross_batch_homog,scProto Stage 2 (scVI),8,0.205,0.133,NaN,NaN,NaN
9,pancreas,cross_batch_homog,SEACells (scVI),8,0.411,0.123,0.0,1.0000,ns
